#### <mark>**Assumptions:**</mark>

1. A workbook contains data for a single Rail Period.
2. All worksheets share the same schema.
3. Required business columns are present.

In [3]:
%run nb_common_utils

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 6, Finished, Available, Finished, True)

In [4]:
# imports
from notebookutils import mssparkutils
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import col
from functools import reduce
from pyspark.sql import DataFrame
import uuid 
from datetime import datetime
from pyspark.sql.types import *

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 7, Finished, Available, Finished, False)

In [5]:
# variables
sourceFilePathIncremental = "Files/Landing"
NAMESPACE = "`GTR Passenger Loading - PROD`.LH_Bronze.dbo"

BRONZE_TABLE = f"{NAMESPACE}.bronze_passenger_loading"
FILE_AUDIT = f"{NAMESPACE}.file_audit"
FILE_WATERMARK = f"{NAMESPACE}.file_watermark"
SILVER_REFRESH_QUEUE = F"`GTR Passenger Loading - PROD`.LH_Silver.dbo.silver_refresh_queue"

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 8, Finished, Available, Finished, False)

In [6]:
def get_excel_files_to_load():
    files = mssparkutils.fs.ls(sourceFilePathIncremental)
    return files

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 9, Finished, Available, Finished, False)

In [7]:
def should_load_workbook(rail_period, source_last_modified_timestamp):

    existing_record = (
        spark.table(FILE_WATERMARK)
        .filter(
            (col("rail_period") == rail_period) &            
            (col("load_status") == "SUCCESS")
        ).first()
        )

    # NOT loaded before
    if existing_record is None:
        return True

    # Source file has changed

    if source_last_modified_timestamp > existing_record["source_last_modified_timestamp"]:
        return True

    # Already loaded
    return False    

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 10, Finished, Available, Finished, False)

In [8]:
def get_rail_period(excel_file_path, sheet_name):
    excel_data = pd.read_excel(excel_file_path,sheet_name=sheet_name,nrows=1)
    rail_period = str(excel_data["Rail period"].iloc[0])
    return rail_period

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 11, Finished, Available, Finished, False)

In [9]:
def write_file_audit(
    file_name,
    rail_period,
    bronze_table_name,
    load_start_timestamp,
    row_count,
    load_status,
    error_message=None
):

    audit_id = str(uuid.uuid4())
    load_end_timestamp = datetime.now()

    schema = StructType([
    StructField("audit_id", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("rail_period", StringType(), True),
    StructField("bronze_table_name", StringType(), True),
    StructField("load_start_timestamp", TimestampType(), True),
    StructField("load_end_timestamp", TimestampType(), True),
    StructField("row_count", LongType(), True),
    StructField("load_status", StringType(), True),
    StructField("error_message", StringType(), True)
])

    audit_data = [{
        "audit_id": audit_id,
        "file_name": file_name,
        "rail_period": str(rail_period),
        "bronze_table_name": bronze_table_name,
        "load_start_timestamp": load_start_timestamp,
        "load_end_timestamp": load_end_timestamp,
        "row_count": row_count,
        "load_status": load_status,
        "error_message": error_message
    }]

    audit_df = spark.createDataFrame(
        audit_data,
        schema=schema
    )

    audit_df.write \
        .mode("append") \
        .saveAsTable(FILE_AUDIT)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 12, Finished, Available, Finished, False)

In [10]:
def load_excel_file_to_data_frame(excel_file_path, rail_period, file_name):

    load_start_timestamp = datetime.now()
    worksheet_dfs = []
    excel_file = pd.ExcelFile(excel_file_path)

    for sheet_name in excel_file.sheet_names:
        print(sheet_name)
        pdf = pd.read_excel(
            excel_file_path,
            sheet_name=sheet_name
            )
        
        #pdf["rail_period"] = str(pdf["Rail period"].iloc[0])
        pdf["source_file_name"] = file_name
        pdf["source_work_sheet_name"] = sheet_name
        pdf["bronze_load_timestamp"] = datetime.now()

        if pdf.empty == False:
            worksheet_dfs.append(
                spark.createDataFrame(pdf)
            )
    
        if not worksheet_dfs:

            write_file_audit(
                file_name=file_name,
                rail_period=rail_period,
                bronze_table_name=BRONZE_TABLE,
                load_start_timestamp=load_start_timestamp,
                row_count=0,
                load_status="SKIPPED",
                error_message=f"No rows found for Rail Period {rail_period}"
            )

            raise Exception(
                f"No rows found for Rail Period {rail_period}"
            )

    bronze_df = reduce(DataFrame.unionByName, worksheet_dfs)

    # sanitizing column names, in case columns has speial characters
    bronze_df = sanitize_column_names(bronze_df)

    if bronze_df.rdd.isEmpty():

        write_file_audit(
            file_name=file_name,
            rail_period=rail_period,
            bronze_table_name=BRONZE_TABLE,
            load_start_timestamp=load_start_timestamp,
            row_count=0,
            load_status="SKIPPED",
            error_message=f"No rows found for Rail Period {rail_period}"
        )

        raise Exception(
            f"No rows found for Rail Period {rail_period}"
        )

    return bronze_df

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 13, Finished, Available, Finished, False)

In [11]:
def update_watermark_table(
    rail_period,
    file_name,
    file_last_modified_date_time
):
    spark.sql(f"""
        MERGE INTO {FILE_WATERMARK} tgt
        USING (
            SELECT
                '{rail_period}' AS rail_period,
                '{file_name}' AS source_file_name,
                TIMESTAMP('{file_last_modified_date_time}') AS source_last_modified_timestamp,
                '{BRONZE_TABLE}' AS bronze_table_name,
                current_timestamp() AS last_loaded_timestamp,
                'SUCCESS' AS load_status
        ) src
        ON tgt.rail_period = src.rail_period

        WHEN MATCHED THEN
        UPDATE SET
            tgt.source_file_name = src.source_file_name,
            tgt.source_last_modified_timestamp = src.source_last_modified_timestamp,
            tgt.bronze_table_name = src.bronze_table_name,
            tgt.last_loaded_timestamp = src.last_loaded_timestamp,
            tgt.load_status = src.load_status

        WHEN NOT MATCHED THEN
        INSERT (
            rail_period,
            source_file_name,
            source_last_modified_timestamp,
            bronze_table_name,
            last_loaded_timestamp,
            load_status
        )
        VALUES (
            src.rail_period,
            src.source_file_name,
            src.source_last_modified_timestamp,
            src.bronze_table_name,
            src.last_loaded_timestamp,
            src.load_status
        )
    """)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 14, Finished, Available, Finished, False)

In [12]:
def queue_silver_refresh(rail_period, load_start_timestamp):

    # Remove any existing active queue entries for this rail period
    spark.sql(f"""
        DELETE FROM {SILVER_REFRESH_QUEUE}
        WHERE rail_period = '{rail_period}'
          AND status IN ('PENDING', 'FAILED')
    """)
    
    queue_schema = StructType([
        StructField("queue_id", StringType(), False),
        StructField("rail_period", StringType(), False),
        StructField("bronze_load_timestamp", TimestampType(), False),
        StructField("processing_start_timestamp", TimestampType(), True),
        StructField("processing_end_timestamp", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)
    ])

    queue_data = [{
        "queue_id": str(uuid.uuid4()),
        "rail_period": str(rail_period),
        "bronze_load_timestamp": load_start_timestamp,
        "processing_start_timestamp": None,
        "processing_end_timestamp": None,
        "status": "PENDING",
        "error_message": None
    }]

    queue_df = spark.createDataFrame(
        queue_data,
        schema=queue_schema
    )

    queue_df.write \
        .mode("append") \
        .saveAsTable(SILVER_REFRESH_QUEUE)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 15, Finished, Available, Finished, False)

In [13]:
def move_file_to_processed(source_file_path):
    """
    Move processed file from Landing to Processed folder.
    """

    try:
        file_name = source_file_path.split("/")[-1]

        processed_file_path = source_file_path.replace(
            "/Files/Landing/",
            "/Files/Processed/"
        )

        print(f"Source      : {source_file_path}")
        print(f"Destination : {processed_file_path}")

        mssparkutils.fs.mv(
            source_file_path,
            processed_file_path
        )

        print(f"Moved: {file_name}")

    except Exception as e:
        print(f"Failed moving file: {source_file_path}")
        raise

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 16, Finished, Available, Finished, False)

In [14]:
def load_file_to_table(
    excel_file_path,
    file_name,
    file_last_modified_date_time
):

    load_start_timestamp = datetime.now()

    try:

        excel_file = pd.ExcelFile(excel_file_path)

        print(
            f"Loading: {file_name} - "
            f"{excel_file.sheet_names}"
        )

        rail_period = get_rail_period(
            excel_file_path,
            excel_file.sheet_names[0]
        )

        if not should_load_workbook(
            rail_period,
            file_last_modified_date_time
        ):
            return

        bronze_df = load_excel_file_to_data_frame(
            excel_file_path,
            rail_period,
            file_name
        )

        row_count = bronze_df.count()

        if row_count == 0:
            raise Exception(
                f"No rows found for Rail Period {rail_period}"
            )


        print("Writting to table.")
        if not spark.catalog.tableExists(BRONZE_TABLE):
            bronze_df.write \
                .mode("overwrite") \
                .saveAsTable(BRONZE_TABLE)
        else:
            # delete existing data
            spark.sql(f"""
                DELETE FROM {BRONZE_TABLE}
                WHERE rail_period = '{rail_period}'
            """)

            # load new data    
            bronze_df.write \
                .mode("append") \
                .saveAsTable(BRONZE_TABLE)        

        print("Update watermark.")
        # update watermark
        update_watermark_table(
            rail_period,
            file_name,
            file_last_modified_date_time
        )
       
        print("Create silver queue.")
        # creating siler queue for processing
        queue_silver_refresh(rail_period,load_start_timestamp)

        print("Move processed file to Processed folder.")
        # move processed files to processed folder
        move_file_to_processed(excel_file_path)

        print("Write audit of success.")
        # audit success
        write_file_audit(
            file_name=file_name,
            rail_period=rail_period,
            bronze_table_name=BRONZE_TABLE,
            load_start_timestamp=load_start_timestamp,
            row_count=row_count,
            load_status="SUCCESS"
        )

    except Exception as ex:

        write_file_audit(
            file_name=file_name,
            rail_period=rail_period if 'rail_period' in locals() else None,
            bronze_table_name=BRONZE_TABLE,
            load_start_timestamp=load_start_timestamp,
            row_count=0,
            load_status="FAILED",
            error_message=str(ex)
        )

        raise

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 17, Finished, Available, Finished, False)

In [15]:
# Bronze data loading

# getting all the files to process
for file in get_excel_files_to_load():
    load_file_to_table(file.path, file.name,datetime.fromtimestamp(file.modifyTime / 1000))


StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 18, Finished, Available, Finished, False)

Loading: 2026-27 P04 GTR Passenger Loading.xlsx - ['Wk1', 'Wk2', 'Wk3', 'Wk4']
Wk1
Wk2
Wk3
Wk4
Writting to table.
Update watermark.
Create silver queue.
Move processed file to Processed folder.
Source      : abfss://6108b3df-b7ad-4249-9b67-d0c7464ff492@onelake.dfs.fabric.microsoft.com/4a8d7df3-030a-46b3-887a-e658d916e17d/Files/Landing/2026-27 P04 GTR Passenger Loading.xlsx
Destination : abfss://6108b3df-b7ad-4249-9b67-d0c7464ff492@onelake.dfs.fabric.microsoft.com/4a8d7df3-030a-46b3-887a-e658d916e17d/Files/Processed/2026-27 P04 GTR Passenger Loading.xlsx
Moved: 2026-27 P04 GTR Passenger Loading.xlsx
Write audit of success.
Loading: 2026-27 P05 GTR Passenger Loading.xlsx - ['Wk1', 'Wk2', 'Wk3', 'Wk4']
Wk1
Wk2
Wk3
Wk4
Writting to table.
Update watermark.
Create silver queue.
Move processed file to Processed folder.
Source      : abfss://6108b3df-b7ad-4249-9b67-d0c7464ff492@onelake.dfs.fabric.microsoft.com/4a8d7df3-030a-46b3-887a-e658d916e17d/Files/Landing/2026-27 P05 GTR Passenger Loading

In [16]:
# temporary cleanup work
# spark.sql("TRUNCATE TABLE LH_Bronze.dbo.file_audit")
# spark.sql("TRUNCATE TABLE LH_Bronze.dbo.file_watermark")
# spark.sql("TRUNCATE TABLE LH_Silver.dbo.silver_refresh_queue")
# spark.sql("TRUNCATE TABLE LH_Bronze.dbo.bronze_passenger_loading")

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 19, Finished, Available, Finished, False)

In [17]:
# moving files back to landing folder from processed.

# processed_files = mssparkutils.fs.ls("Files/Processed")

# for file in processed_files:

#     destination = file.path.replace(
#         "/Files/Processed/",
#         "/Files/Landing/"
#     )

#     mssparkutils.fs.mv(
#         file.path,
#         destination,
#         True
#     )

#     print(f"Moved {file.name}")

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 20, Finished, Available, Finished, False)

In [19]:
# check file audit

df = spark.read.table("LH_Bronze.dbo.file_audit")

display(df)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1d7ad40d-47e9-499b-be3f-11d126cceb10)

In [20]:
# check file watermark
df = spark.read.table("LH_Bronze.dbo.file_watermark")
display(df)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4038298b-edc4-4d1d-9370-854a64f31c49)

In [23]:
# silver queue check

df = spark.read.table("LH_Silver.dbo.silver_refresh_queue")
display(df)

StatementMeta(, 55399c47-5ccc-4a10-8661-be28ad8ce17e, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d55c2cc3-d1ae-42da-bad3-4c42584bf209)